# pyTelops Showcase

Demonstrates the full workflow of the pyTelops driver for Telops FAST M3k thermal cameras:

1. Discovery & connection
2. Camera configuration
3. Live streaming
4. Buffer recording (5s at 2000 fps = 10,000 frames)
5. Buffer download
6. Viewing saved images

In [ ]:
# Firewall: allow inbound UDP for this python.exe (needed for GVSP streaming)
# Run once — requires admin. If already done, this cell is a no-op.
import sys, subprocess
exe = sys.executable
rule_name = "pyTelops-GVSP"

# Check if rule already exists
result = subprocess.run(
    ["netsh", "advfirewall", "firewall", "show", "rule", f"name={rule_name}"],
    capture_output=True, text=True)

if "No rules match" in result.stderr or result.returncode != 0:
    print(f"Adding firewall rule for: {exe}")
    r = subprocess.run(
        ["netsh", "advfirewall", "firewall", "add", "rule",
         f"name={rule_name}", "dir=in", "action=allow",
         "protocol=UDP", f"program={exe}"],
        capture_output=True, text=True)
    if r.returncode == 0:
        print("Firewall rule added.")
    else:
        print(f"Failed (need admin?): {r.stderr.strip()}")
else:
    print(f"Firewall rule '{rule_name}' already exists.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time

from pyTelops import Camera, discover
from pyTelops import registers as reg

## 1. Discovery

Find all Telops cameras on the network via GVCP broadcast.

In [ ]:
cameras = discover()
for c in cameras:
    print(f"{c['manufacturer']} {c['model']} at {c['ip']}")
    print(f"  Serial: {c.get('serial', '?')}")
    print(f"  Version: {c.get('device_version', '?')}")

## 2. Connect

In [ ]:
cam = Camera()
cam.connect()
print(f"Connected: {cam}")
print(f"State: {cam.state}")

## 3. Camera info & configuration

In [ ]:
# Current configuration
info = cam.info
for key, val in info.items():
    print(f"  {key:20s}: {val}")

In [ ]:
# Clear any leftover buffer state (locks calibration mode, acquisition, etc.)
cam.stop_stream()
cam.buffer_clear()

# Configure for 2000 fps measurement
cam.frame_rate = 2000.0
cam.exposure = 30.0                           # 30 us (must be < 1/2000 = 500 us)
cam.calibration_mode = reg.CalibrationMode.RT  # radiometric temperature

print(f"Frame rate:  {cam.frame_rate:.0f} Hz")
print(f"Exposure:    {cam.exposure:.1f} us")
print(f"Calibration: {cam.calibration_mode.name}")
print(f"Resolution:  {cam.resolution}")
print(f"Temperature: {cam.temperature:.1f} C")

## 4. Live streaming

Grab a single frame and display it.

In [ ]:
frame = cam.grab()
print(f"Frame shape: {frame.shape}, dtype: {frame.dtype}")
print(f"Min: {frame.min()}, Max: {frame.max()}, Std: {frame.std():.0f}")

# Skip 2 header rows
img = frame[cam.HEADER_ROWS:, :]

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(img, cmap='inferno')
ax.set_title('Single frame')
plt.colorbar(im, ax=ax, label='Raw counts')
plt.tight_layout()
plt.show()

## 5. Live viewer (interactive)

Opens a Tkinter window with real-time thermal display and colormap selector.
Close the window to continue.

In [ ]:
# Uncomment to launch live viewer (blocks until window is closed)
# cam.live_view(colormap='inferno', scale=2)

## 6. Configure buffer for 5s at 2000 fps

The camera has a 16 GB internal ring buffer that records at full sensor speed,
independent of the Ethernet link.

In [ ]:
DURATION = 5.0   # seconds
FPS = 2000       # frames per second
N_FRAMES = int(DURATION * FPS)

print(f"Recording: {N_FRAMES} frames ({DURATION}s at {FPS} fps)")
print(f"Data size: {N_FRAMES * 165120 / 1e6:.0f} MB")

cam.buffer_configure(
    n_sequences=1,
    frames_per_seq=N_FRAMES,
    pre_moi=0,
    moi_source=reg.MemoryBufferMOISource.SOFTWARE,
)
print(f"Buffer status: {cam.buffer_status().name}")

## 7. Record to buffer

Arm the camera, wait for recording to fill, then fire the MOI trigger.

In [ ]:
# Arm and start acquisition
cam.write_register(reg.REG_ACQUISITION_ARM, 1)
cam.write_register(reg.REG_ACQUISITION_START, 1)
print(f"Status: {cam.buffer_status().name}")

# Fire software MOI trigger
cam.buffer_fire_moi()
print("MOI fired, recording...")

# Wait for recording to finish (timeout after 30s)
t0 = time.monotonic()
TIMEOUT = 30.0
while time.monotonic() - t0 < TIMEOUT:
    status = cam.buffer_status()
    elapsed = time.monotonic() - t0
    print(f"  {elapsed:.1f}s — {status.name}", flush=True)
    if status in (reg.MemoryBufferStatus.HOLDING, reg.MemoryBufferStatus.IDLE):
        break
    time.sleep(1.0)
else:
    print(f"WARNING: Timed out after {TIMEOUT}s!")

cam.write_register(reg.REG_ACQUISITION_STOP, 1)
elapsed = time.monotonic() - t0

recorded = cam.buffer_recorded_frames(0)
print(f"\nRecorded {recorded} frames in {elapsed:.1f}s")
print(f"Status: {cam.buffer_status().name}")

## 8. Download from buffer

Download at max speed (bitrate=1000 Mbps, packet_size=9000).

In [ ]:
t0 = time.monotonic()
data = cam.buffer_download(sequence=0)
elapsed = time.monotonic() - t0

print(f"Shape: {data.shape}, dtype: {data.dtype}")
print(f"Time: {elapsed:.1f}s")
print(f"Speed: {data.shape[0] / elapsed:.0f} fps, "
      f"{data.shape[0] * data.shape[1] * data.shape[2] * 2 / elapsed / 1e6:.1f} MB/s")

In [ ]:
# Save to disk
np.save('measurement_2000fps_5s.npy', data)
print(f"Saved: measurement_2000fps_5s.npy ({data.nbytes / 1e6:.0f} MB)")

In [ ]:
# Clean up buffer
cam.buffer_clear()

## 9. View saved images

Browse the downloaded frames.

In [ ]:
# Load (or use data already in memory)
# data = np.load('measurement_2000fps_5s.npy')

# Strip header rows
images = data[:, cam.HEADER_ROWS:, :]
n, h, w = images.shape
print(f"{n} frames, {h}x{w} pixels")
print(f"Duration: {n / FPS:.2f}s at {FPS} fps")

In [ ]:
# Show first, middle, and last frame
indices = [0, n // 2, n - 1]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, idx in zip(axes, indices):
    im = ax.imshow(images[idx], cmap='inferno')
    ax.set_title(f'Frame {idx} (t={idx/FPS:.3f}s)')
    plt.colorbar(im, ax=ax, shrink=0.8)

plt.tight_layout()
plt.show()

In [ ]:
# Temporal statistics: mean and std across all frames
mean_img = images.mean(axis=0)
std_img = images.std(axis=0)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

im0 = axes[0].imshow(mean_img, cmap='inferno')
axes[0].set_title('Temporal mean')
plt.colorbar(im0, ax=axes[0], shrink=0.8)

im1 = axes[1].imshow(std_img, cmap='hot')
axes[1].set_title('Temporal std (dynamic content)')
plt.colorbar(im1, ax=axes[1], shrink=0.8)

plt.tight_layout()
plt.show()

In [ ]:
# Time series of a single pixel
py, px = h // 2, w // 2  # center pixel
t = np.arange(n) / FPS

fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(t, images[:, py, px], linewidth=0.5)
ax.set_xlabel('Time [s]')
ax.set_ylabel('Raw counts')
ax.set_title(f'Center pixel ({px}, {py}) time series')
ax.set_xlim(0, t[-1])
plt.tight_layout()
plt.show()

## 10. Disconnect

In [ ]:
cam.disconnect()
print(f"State: {cam.state}")